# 🇻🇳 Phân Loại Rác Thải Việt Nam - Training trên Google Colab

Notebook này được thiết kế để huấn luyện mô hình **Baseline CNN** (hoặc Transfer Learning) phân loại rác thải trên môi trường GPU miễn phí của Google Colab.

### ⚡ Hướng dẫn nhanh:
1. Vào **Runtime** ➔ **Change runtime type** ➔ Chọn **T4 GPU** ➔ **Save**.
2. Nhấn **Runtime** ➔ **Run all** (hoặc chạy từng cell bằng `Shift + Enter`).
3. Sau khi train 30 epochs (mất khoảng **1-2 phút** trên GPU), cell cuối cùng sẽ tải trực tiếp file `best_model.pth` về máy tính của bạn.
4. Bạn chỉ cần copy `best_model.pth` vào thư mục `models/` của project trên máy để chạy Gradio Web App!

## 1. Kiểm tra cấu hình GPU Colab

In [ ]:
!nvidia-smi

## 2. Cài đặt thư viện phụ trợ

In [ ]:
!pip install -q pyyaml scikit-learn seaborn matplotlib tqdm pillow

## 3. Tải và chuẩn bị bộ dữ liệu TrashNet

In [ ]:
import os, urllib.request, zipfile, shutil
from pathlib import Path

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
CLASSES = ["plastic", "paper", "metal", "glass", "organic", "styrofoam"]

for c in CLASSES:
    (RAW_DIR / c).mkdir(parents=True, exist_ok=True)

print("📥 Đang tải dataset TrashNet từ GitHub...")
url = "https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip"
zip_path = "dataset-resized.zip"
urllib.request.urlretrieve(url, zip_path)

print("📦 Đang giải nén...")
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("data_temp")

# Ánh xạ các nhãn của TrashNet sang phân loại Việt Nam
mapping = {
    "cardboard": "paper",
    "glass": "glass",
    "metal": "metal",
    "paper": "paper",
    "plastic": "plastic",
    "trash": "organic",
}

src_dir = Path("data_temp/dataset-resized")
for orig_class, target_class in mapping.items():
    class_folder = src_dir / orig_class
    if class_folder.exists():
        for img in class_folder.glob("*.jpg"):
            shutil.copy2(img, RAW_DIR / target_class / f"{orig_class}_{img.name}")

shutil.rmtree("data_temp")
os.remove(zip_path)

print("\n📊 Phân bố ảnh đã tải:")
for c in CLASSES:
    cnt = len(list((RAW_DIR / c).glob("*.jpg")))
    print(f"  - {c:10s}: {cnt} ảnh")

## 4. Làm sạch dữ liệu và Phân chia tập Train / Val / Test (70% - 15% - 15%)

In [ ]:
import hashlib
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split

PROCESSED_DIR = DATA_DIR / "processed"
for c in CLASSES:
    (PROCESSED_DIR / c).mkdir(parents=True, exist_ok=True)

seen_hashes = set()
records = []

# Lọc ảnh hỏng và ảnh trùng MD5, resize về 224x224 chuẩn
for label_idx, c in enumerate(CLASSES):
    for img_path in (RAW_DIR / c).glob("*.jpg"):
        try:
            with Image.open(img_path) as img:
                img.verify()
            with open(img_path, "rb") as f:
                h = hashlib.md5(f.read()).hexdigest()
            if h in seen_hashes:
                continue
            seen_hashes.add(h)
            
            with Image.open(img_path) as img:
                img_rgb = img.convert("RGB").resize((224, 224), Image.LANCZOS)
                save_path = PROCESSED_DIR / c / img_path.name
                img_rgb.save(save_path, "JPEG", quality=95)
                records.append({"filepath": str(save_path), "class_name": c, "label": label_idx})
        except Exception:
            continue

df = pd.DataFrame(records)
train_val_df, test_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df["label"])
train_df, val_df = train_test_split(train_val_df, test_size=0.1765, random_state=42, stratify=train_val_df["label"]) # ~15% total

print(f"✓ Train set: {len(train_df)} ảnh")
print(f"✓ Val set:   {len(val_df)} ảnh")
print(f"✓ Test set:  {len(test_df)} ảnh")

## 5. Định nghĩa Dataset & Data Augmentation

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class WasteDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, row["label"]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(WasteDataset(train_df, train_transform), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(WasteDataset(val_df, val_transform), batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(WasteDataset(test_df, val_transform), batch_size=32, shuffle=False, num_workers=2)

## 6. Kiến trúc Baseline CNN

In [ ]:
import torch.nn as nn

class BaselineCNN(nn.Module):
    """Baseline CNN: 4 Conv Blocks + Global Pooling + Classifier"""
    def __init__(self, num_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Thiết bị được chọn: {device}")

## 7. Huấn luyện Baseline Model (30 Epochs trên GPU Colab)

In [ ]:
import torch.optim as optim
from sklearn.metrics import f1_score
import numpy as np
import time

model = BaselineCNN(num_classes=6).to(device)

# Tính class weights xử lý mất cân bằng nhãn
class_counts = train_df["label"].value_counts().sort_index()
total_samples = len(train_df)
weights = [total_samples / (6 * class_counts.get(i, 1)) for i in range(6)]
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

EPOCHS = 30
best_val_f1 = 0.0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_f1": []}

start_time = time.time()
print("🚀 Bắt đầu huấn luyện...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
        _, pred = torch.max(out, 1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    scheduler.step()
    
    train_loss = t_loss / total
    train_acc = correct / total
    
    model.eval()
    v_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            v_loss += criterion(out, labels).item() * imgs.size(0)
            _, pred = torch.max(out, 1)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    val_loss = v_loss / len(all_labels)
    val_acc = np.mean(np.array(all_preds) == np.array(all_labels))
    val_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)
    
    save_msg = ""
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_f1": val_f1,
            "val_acc": val_acc,
            "class_names": CLASSES,
            "model_type": "baseline"
        }, "best_model.pth")
        save_msg = "💾 Saved Best!"
        
    print(f"Epoch {epoch:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f} {save_msg}")

elapsed = (time.time() - start_time) / 60
print(f"\n🎉 Hoàn thành trong {elapsed:.2f} phút! Best Val F1: {best_val_f1:.4f}")

## 8. Đánh giá trên tập Test & Xuất Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Tải checkpoint tốt nhất
ckpt = torch.load("best_model.pth", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        _, pred = torch.max(out, 1)
        test_preds.extend(pred.cpu().numpy())
        test_labels.extend(labels.numpy())

print("=== BÁO CÁO KẾT QUẢ ĐÁNH GIÁ (TEST SET) ===\n")
active_labels = sorted(list(set(test_labels)))
active_names = [CLASSES[i] for i in active_labels]
print(classification_report(test_labels, test_preds, target_names=active_names, zero_division=0))

# Vẽ biểu đồ Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=active_names, yticklabels=active_names)
plt.title("Confusion Matrix - Baseline CNN (Test Set)")
plt.ylabel("Nhãn thực tế (True Label)")
plt.xlabel("Nhãn dự đoán (Predicted Label)")
plt.tight_layout()
plt.show()

## 9. Tải file trọng số `best_model.pth` về máy tính

In [ ]:
from google.colab import files
print("📥 Đang kích hoạt tải file best_model.pth về máy tính của bạn...")
files.download("best_model.pth")